# Lens Notebook

This notebook shows how to use both `LogitLens` and `TunedLens` with `ModelWithSplitPoints` across several model families.

Origins:
- Logit Lens: nostalgebraist, *Interpreting GPT: the logit lens*
- Related vocabulary-space analysis: Geva et al. (2022)
- Tuned Lens: Belrose et al. (2023)

The tuned lens implementation supports three initialization modes:
- `logit_lens`: identity initialization so tuning starts from the plain logit-lens behavior
- `xavier`: Xavier uniform initialization with zero bias
- `default`: the current `torch.nn.Linear` initialization

The notebook uses tiny checkpoints so it stays light enough for quick experimentation.
Some of them are random checkpoints, so semantic quality is not the goal here: the examples are mainly meant to illustrate the API and the decodability metrics.

Metric interpretation:
- `mean_target_probability`: higher is better
- `target_cross_entropy`: lower is better
- `perplexity`: lower is better for causal language models
- `kl_divergence_to_model`: lower is better and differentiable, which makes it useful as a regularization target for linear decodability
- `model_top1_agreement`: agreement with the final model argmax

The raw `explain()` and `lens()` outputs are tensor-first and use `top_indices` / `top_scores`.
Human-readable decoding is handled by the visualization layer.


In [1]:
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

from interpreto import LogitLens, ModelWithSplitPoints, TunedLens


def summarize_metrics(metrics, split_point):
    keys = [
        "target_source",
        "nb_evaluated_elements",
        "mean_target_probability",
        "target_cross_entropy",
        "target_accuracy",
        "mean_max_probability",
        "kl_divergence_to_model",
        "model_top1_agreement",
        "perplexity",
    ]
    return {key: metrics[split_point][key] for key in keys if key in metrics[split_point]}

/gpfs/users/bernasra/.conda/envs/ruche-py311-sci-torch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Causal Language Model

We start with a small GPT-style model and inspect two prompts at once.


In [2]:
causal_model_name = "hf-internal-testing/tiny-random-gpt2"
causal_model = AutoModelForCausalLM.from_pretrained(causal_model_name)
causal_tokenizer = AutoTokenizer.from_pretrained(causal_model_name)
if causal_tokenizer.pad_token is None:
    causal_tokenizer.pad_token = causal_tokenizer.eos_token

causal_model_with_split_points = ModelWithSplitPoints(
    causal_model,
    tokenizer=causal_tokenizer,
    split_point="transformer.h.1.mlp",
    batch_size=2,
    device_map="cpu",
)

causal_examples = [
    "Interpreto is useful.",
    "Interpreto helps explain models.",
]

causal_model_with_split_points.split_point

Loading weights: 100%|██████████| 64/64 [00:00<00:00, 7482.73it/s]


'transformer.h.1.mlp'

In [3]:
causal_logit_lens = LogitLens(causal_model_with_split_points, top_k=3)
causal_logit_explanations = causal_logit_lens.explain(causal_examples)
causal_logit_lens.lens(causal_examples)

{'transformer.h.1.mlp': {'top_indices': tensor([[[962, 929, 551],
           [641, 591,  26],
           [811, 271, 782],
           [810, 549, 171],
           [ 57, 738, 529],
           [204, 818, 411],
           [203, 400, 214],
           [949,  64, 128],
           [204, 145, 297],
           [456, 964, 816],
           [275,  54, 995],
           [325, 598, 111],
           [638,  11, 211],
           [456, 816,  71],
           [211, 259, 718],
           [292, 283, 341],
           [614, 375, 291],
           [558, 637, 380]],
  
          [[203, 204,  29],
           [128, 585, 633],
           [436, 204, 675],
           [866, 204, 810],
           [871, 669, 386],
           [145, 205, 204],
           [766, 214, 633],
           [458, 885, 336],
           [990, 633, 474],
           [456, 964, 816],
           [995, 591, 384],
           [598, 900, 783],
           [832, 870,  11],
           [949, 483, 421],
           [832,  54, 212],
           [995, 248, 238],
      

In [4]:
(
    causal_logit_explanations["transformer.h.1.mlp"]["top_indices"][0, 0],
    causal_logit_explanations["transformer.h.1.mlp"]["top_scores"][0, 0],
)

(tensor([962, 929, 551]), tensor([0.0014, 0.0014, 0.0013]))

In [5]:
causal_logit_metrics = causal_logit_lens.metrics(causal_examples)
summarize_metrics(causal_logit_metrics, "transformer.h.1.mlp")

{'target_source': 'next_token',
 'nb_evaluated_elements': 29,
 'mean_target_probability': 0.0010038625914603472,
 'target_cross_entropy': 6.910143852233887,
 'target_accuracy': 0.0,
 'mean_max_probability': 0.0014180837897583842,
 'kl_divergence_to_model': 0.008136783726513386,
 'model_top1_agreement': 0.03448275849223137,
 'perplexity': 1002.3914184570312}

## Masked Language Model

The same `LogitLens` workflow also works on masked-language-model checkpoints.


In [6]:
masked_model_name = "hf-internal-testing/tiny-random-bert"
masked_config = AutoConfig.from_pretrained(masked_model_name)
masked_model = AutoModelForMaskedLM.from_config(masked_config)
masked_tokenizer = AutoTokenizer.from_pretrained(masked_model_name)

masked_model_with_split_points = ModelWithSplitPoints(
    masked_model,
    tokenizer=masked_tokenizer,
    split_point="bert.encoder.layer.1.output",
    batch_size=2,
    device_map="cpu",
)

masked_examples = [
    "Interpreto is useful",
    "Interpreto explains transformers",
]

masked_model_with_split_points.split_point

'bert.encoder.layer.1.output'

In [7]:
masked_logit_lens = LogitLens(masked_model_with_split_points, top_k=4)
masked_logit_explanations = masked_logit_lens.explain(masked_examples)
masked_logit_lens.lens(masked_examples)

{'bert.encoder.layer.1.output': {'top_indices': tensor([[[1012,  252,  342,  249],
           [ 620,  838,  783,  837],
           [ 622,  458, 1016,  620],
           [ 580,  313,  314,  647],
           [  70,  748,  620,  299],
           [1066, 1106,  548,  135],
           [ 975,  210, 1066,  208],
           [1066,  816,  610,  158],
           [ 708,  962,  254,  800],
           [1103,  758, 1012,  485],
           [ 458,  329,  891,  628],
           [ 779,  620,  208,  776],
           [ 153,  752,  222,  881],
           [  54,  610,  342, 1086],
           [ 420,  425,  240,  478],
           [ 620,  975,  748,  615],
           [ 533,  282,  962,  656],
           [ 975,  779,  485,  588],
           [ 222,  761,  652,  754],
           [ 458,   86,  179, 1034],
           [ 921,  635,  210,  975],
           [ 610, 1086,  579, 1032],
           [ 610,  858,  981,  236],
           [ 387,  689,  588,  458],
           [ 222,   13,  720,  898],
           [1079,  222,  858,

In [8]:
masked_logit_metrics = masked_logit_lens.metrics(masked_examples)
summarize_metrics(masked_logit_metrics, "bert.encoder.layer.1.output")

{'target_source': 'token_identity',
 'nb_evaluated_elements': 48,
 'mean_target_probability': 0.0008483824203722179,
 'target_cross_entropy': 7.0776848793029785,
 'target_accuracy': 0.0,
 'mean_max_probability': 0.0012879521818831563,
 'kl_divergence_to_model': 1.8258966747453087e-06,
 'model_top1_agreement': 0.9791666865348816}

## Sequence Classification

For classification, the same framework exposes intermediate label distributions and classification-oriented scores.
To keep this notebook lightweight and warning-free, the example below builds a small two-label classifier from the tiny BERT configuration instead of loading mismatched task heads from a generic checkpoint.


In [9]:
classification_config = AutoConfig.from_pretrained(masked_model_name)
classification_config.num_labels = 2
classification_config.id2label = {0: "negative", 1: "positive"}
classification_config.label2id = {"negative": 0, "positive": 1}
classification_model = AutoModelForSequenceClassification.from_config(classification_config)
classification_tokenizer = AutoTokenizer.from_pretrained(masked_model_name)

classification_model_with_split_points = ModelWithSplitPoints(
    classification_model,
    tokenizer=classification_tokenizer,
    split_point="bert.encoder.layer.1.output",
    batch_size=2,
    device_map="cpu",
)

classification_examples = [
    "Interpreto is helpful",
    "Interpreto is practical",
]
classification_label_names = {0: "negative", 1: "positive"}
classification_targets = [1, 0]

classification_model_with_split_points.split_point

'bert.encoder.layer.1.output'

In [10]:
classification_logit_lens = LogitLens(classification_model_with_split_points, top_k=2)
classification_logit_explanations = classification_logit_lens.explain(classification_examples)
classification_logit_lens.lens(classification_examples, label_names=classification_label_names)

{'bert.encoder.layer.1.output': {'top_indices': tensor([[0, 1],
          [0, 1]]),
  'top_scores': tensor([[0.5018, 0.4982],
          [0.5018, 0.4982]])}}

In [11]:
classification_logit_metrics = classification_logit_lens.metrics(
    classification_examples,
    targets=classification_targets,
)
summarize_metrics(classification_logit_metrics, "bert.encoder.layer.1.output")

{'target_source': 'provided_targets',
 'nb_evaluated_elements': 2,
 'mean_target_probability': 0.5,
 'target_cross_entropy': 0.6931533217430115,
 'target_accuracy': 0.5,
 'mean_max_probability': 0.501756489276886,
 'kl_divergence_to_model': -1.4901161193847656e-08,
 'model_top1_agreement': 1.0}

## Tuned Lens On A Small Dataset

The final section fits a `TunedLens` on a tiny text collection for the causal model.
This is only a small demonstration, but it shows how the decodability metrics can be tracked before and after tuning.


In [12]:
supported_modes = ["logit_lens", "xavier", "default"]
[
    TunedLens(causal_model_with_split_points, top_k=3, initialization_mode=mode).initialization_mode
    for mode in supported_modes
]

['logit_lens', 'xavier', 'default']

In [13]:
tuning_texts = [
    "Interpreto is useful.",
    "Interpreto helps explain transformers.",
    "Interpreto makes debugging easier.",
    "Interpreto is practical for analysis.",
]

tuned_lens = TunedLens(causal_model_with_split_points, top_k=3, initialization_mode="logit_lens")
pre_tuning_metrics = summarize_metrics(tuned_lens.metrics(causal_examples), "transformer.h.1.mlp")
history = tuned_lens.fit(tuning_texts, epochs=2, batch_size=2)
post_tuning_metrics = summarize_metrics(tuned_lens.metrics(causal_examples), "transformer.h.1.mlp")
history

{'loss': [0.008252160623669624, 0.00788214709609747],
 'split_point': 'transformer.h.1.mlp',
 'epochs': 2}

In [14]:
{"before": pre_tuning_metrics, "after": post_tuning_metrics}

{'before': {'target_source': 'next_token',
  'nb_evaluated_elements': 29,
  'mean_target_probability': 0.0010038625914603472,
  'target_cross_entropy': 6.910143852233887,
  'target_accuracy': 0.0,
  'mean_max_probability': 0.0014180837897583842,
  'kl_divergence_to_model': 0.008136783726513386,
  'model_top1_agreement': 0.03448275849223137,
  'perplexity': 1002.3914184570312},
 'after': {'target_source': 'next_token',
  'nb_evaluated_elements': 29,
  'mean_target_probability': 0.0010151939932256937,
  'target_cross_entropy': 6.898152828216553,
  'target_accuracy': 0.0,
  'mean_max_probability': 0.001416579121723771,
  'kl_divergence_to_model': 0.007438413333147764,
  'model_top1_agreement': 0.03448275849223137,
  'perplexity': 990.4434814453125}}

In [15]:
tuned_lens_examples = [
    "Interpreto helps debug transformers.",
    "Interpreto makes analysis practical.",
]
tuned_lens_explanations = tuned_lens.explain(tuned_lens_examples)
tuned_lens.lens(tuned_lens_examples)

{'transformer.h.1.mlp': {'top_indices': tensor([[[929, 551, 418],
           [688, 452, 325],
           [128, 116,  39],
           [204, 240, 297],
           [178, 460, 865],
           [642, 430, 949],
           [633, 145, 203],
           [446, 637, 738],
           [204, 458, 282],
           [807, 164, 248],
           [832,  54, 995],
           [716, 738, 173],
           [753,  11, 231],
           [949, 310, 164],
           [591, 568, 792],
           [248, 665,  73],
           [323, 669,  61],
           [665, 178, 741],
           [672, 172, 248]],
  
          [[203,  29, 204],
           [128, 585, 275],
           [436, 297,  39],
           [810, 319, 204],
           [871, 782, 669],
           [145, 205, 204],
           [ 78, 608, 207],
           [275, 870,  54],
           [607, 914, 267],
           [591, 984,  71],
           [995, 832, 964],
           [173, 870, 558],
           [870, 939, 843],
           [ 65, 995, 930],
           [591,  26, 632],
      

In [16]:
held_out_metrics = tuned_lens.metrics(
    [
        "Interpreto helps debug transformers.",
        "Interpreto makes analysis practical.",
    ]
)
summarize_metrics(held_out_metrics, "transformer.h.1.mlp")

{'target_source': 'next_token',
 'nb_evaluated_elements': 36,
 'mean_target_probability': 0.001018359325826168,
 'target_cross_entropy': 6.8959479331970215,
 'target_accuracy': 0.0,
 'mean_max_probability': 0.001417913706973195,
 'kl_divergence_to_model': 0.00745380250737071,
 'model_top1_agreement': 0.0,
 'perplexity': 988.2620849609375}